In [2]:
#!/usr/bin/env python3
"""
Diagnostic script to inspect and validate preprocessed Europarl PKL/JSON datasets.
"""

import json
import pickle
from pathlib import Path

# Paths
PKL_PATH = Path("data/train/europarl/train_en_pt.pkl")
VOCAB_PATH = Path("data/train/europarl/vocab_multilingual.json")


def load_vocab(vocab_path: Path):
    if not vocab_path.exists():
        raise FileNotFoundError(f"Vocabulary file not found: {vocab_path}")

    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)

    token_to_idx = vocab["token_to_idx"]
    # Handle integer keys safely
    idx_to_token = {int(v): k for k, v in token_to_idx.items()}
    return token_to_idx, idx_to_token


def decode_ids(ids, idx_to_token):
    return " ".join(idx_to_token.get(int(i), f"<UNK:{i}>") for i in ids)


def main():
    token_to_idx, idx_to_token = load_vocab(VOCAB_PATH)

    # Determine target language tag dynamically from path (e.g., train_en_pt.pkl -> <PT>)
    filename = PKL_PATH.stem.lower()
    target_lang = filename.split("_")[-1]  # "pt", "es", "fr", or "en"
    target_lang_tag = f"<{target_lang.upper()}>"

    print("=" * 80)
    print(f"Inspecting Dataset: {PKL_PATH}")
    print(f"Target Language Tag: {target_lang_tag}")
    print("=" * 80)

    # Load dataset
    if not PKL_PATH.exists():
        raise FileNotFoundError(f"Dataset file not found: {PKL_PATH}")

    with open(PKL_PATH, "rb") as f:
        data = pickle.load(f)

    # Metadata overview
    total_samples = len(data)
    print(f"Total samples      : {total_samples:,}")
    print(f"Data structure     : {type(data)}")
    if total_samples > 0:
        print(f"First item type    : {type(data[0])}")
        print(f"First item length  : {len(data[0]) if isinstance(data[0], (tuple, list)) else 'N/A'}")

    # Inspect Sample #2
    sample_idx = min(2, total_samples - 1)
    src_sample, trg_sample = data[sample_idx]

    print("\n" + "-" * 80)
    print(f"Sample #{sample_idx} Raw Token IDs:")
    print("SRC:", src_sample[:15], "..." if len(src_sample) > 15 else "")
    print("TRG:", trg_sample[:15], "..." if len(trg_sample) > 15 else "")

    print("\nSample Decoded Text:")
    print("SRC:", decode_ids(src_sample, idx_to_token))
    print("TRG:", decode_ids(trg_sample, idx_to_token))
    print("-" * 80 + "\n")

    # Required Token IDs
    start_idx = token_to_idx.get("<START>")
    end_idx = token_to_idx.get("<END>")
    en_idx = token_to_idx.get("<EN>")
    trg_idx = token_to_idx.get(target_lang_tag)

    if trg_idx is None:
        print(f"⚠️ Warning: Target language token '{target_lang_tag}' not found in vocabulary!")

    # Check Dataset Integrity
    errors = 0
    checks_passed = 0
    num_to_check = min(5000, total_samples)  # Validate up to 5,000 samples

    src_lengths = []
    trg_lengths = []
    vocab_size = len(token_to_idx)

    for i in range(num_to_check):
        item = data[i]

        # 1. Structure check
        if not isinstance(item, (tuple, list)) or len(item) != 2:
            print(f"❌ Bad item format at index {i}: Expected tuple/list of 2 elements.")
            errors += 1
            continue

        src, trg = item

        # 2. Empty sequence check
        if len(src) < 3 or len(trg) < 3:
            print(f"❌ Sequence too short at index {i}: len(src)={len(src)}, len(trg)={len(trg)}")
            errors += 1

        # 3. <START> tag check
        if src[0] != start_idx:
            print(f"❌ Missing <START> in SRC at index {i}")
            errors += 1
        if trg[0] != start_idx:
            print(f"❌ Missing <START> in TRG at index {i}")
            errors += 1

        # 4. Language tag check
        if src[1] != en_idx:
            print(f"❌ Incorrect SRC language tag at index {i}. Expected <EN> ({en_idx}), got {src[1]}")
            errors += 1

        if trg_idx is not None and trg[1] != trg_idx:
            print(f"❌ Incorrect TRG language tag at index {i}. Expected {target_lang_tag} ({trg_idx}), got {trg[1]}")
            errors += 1

        # 5. <END> tag check
        if src[-1] != end_idx:
            print(f"❌ Missing <END> in SRC at index {i}")
            errors += 1
        if trg[-1] != end_idx:
            print(f"❌ Missing <END> in TRG at index {i}")
            errors += 1

        # 6. Vocab index bounds check
        if any(idx >= vocab_size or idx < 0 for idx in src):
            print(f"❌ Token ID out of vocab range in SRC at index {i}")
            errors += 1
        if any(idx >= vocab_size or idx < 0 for idx in trg):
            print(f"❌ Token ID out of vocab range in TRG at index {i}")
            errors += 1

        src_lengths.append(len(src))
        trg_lengths.append(len(trg))
        checks_passed += 1

    # Statistical Overview
    if src_lengths and trg_lengths:
        print("Sequence Length Statistics (Checked Samples):")
        print(f"  SRC -> Min: {min(src_lengths)}, Max: {max(src_lengths)}, Avg: {sum(src_lengths)/len(src_lengths):.2f}")
        print(f"  TRG -> Min: {min(trg_lengths)}, Max: {max(trg_lengths)}, Avg: {sum(trg_lengths)/len(trg_lengths):.2f}")

    # JSON Parity Check
    json_path = PKL_PATH.with_suffix(".json")
    if json_path.exists():
        print("\nChecking parity with corresponding JSON file...")
        with open(json_path, "r", encoding="utf-8") as f:
            json_data = json.load(f)

        if len(json_data) != len(data):
            print(f"❌ Length mismatch: PKL has {len(data):,} items, but JSON has {len(json_data):,} items.")
            errors += 1
        elif json_data[sample_idx] != list(data[sample_idx]):
            print("❌ Content mismatch between PKL and JSON files at sample #2.")
            errors += 1
        else:
            print("✅ JSON file matches PKL dataset.")

    # Result Summary
    print("\n" + "=" * 80)
    print(f"Verification Results ({num_to_check:,} samples checked):")
    if errors == 0:
        print("✅ Dataset passed all validation checks!")
    else:
        print(f"❌ Found {errors:,} error(s) in dataset.")
    print("=" * 80)


if __name__ == "__main__":
    main()

Inspecting Dataset: data/train/europarl/train_en_pt.pkl
Target Language Tag: <PT>
Total samples      : 47,209
Data structure     : <class 'list'>
First item type    : <class 'tuple'>
First item length  : 2

--------------------------------------------------------------------------------
Sample #2 Raw Token IDs:
SRC: [1, 4, 130, 80, 9, 53, 15197, 9, 31, 395, 23, 398, 14368, 8, 1997] ...
TRG: [1, 5, 832, 9, 52, 118, 15197, 9, 15922, 141, 3860, 10, 1569, 3402, 8] ...

Sample Decoded Text:
SRC: <START> <EN> thank you , mr segni , i shall do so gladly . indeed , it is quite in keeping with the positions this house has always adopted . <END>
TRG: <START> <PT> obrigada , senhor deputado segni , fá lo ei de boa vontade . com efeito , essa é a linha das posições que o nosso parlamento sempre adoptou . <END>
--------------------------------------------------------------------------------

Sequence Length Statistics (Checked Samples):
  SRC -> Min: 7, Max: 33, Avg: 16.77
  TRG -> Min: 7, Max: 33,